# Demo: Segregation Index (SI) computation

This notebook demonstrates the core pipeline from the manuscript:
*Synaptic compartment polarity and connectivity in the adult Drosophila whole-brain connectome*

It runs on a sample of **200 neurons** with a subset of the Princeton synapse table.

**To run locally:** install dependencies (`pip install navis`) and run cells in order from the `demo/` directory.

**To run on Colab:** click the badge in the repository README — all data is cloned automatically.

---

In [ ]:
# Install navis if not already installed
# Uncomment the line below if running for the first time
# !pip install navis

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import sys

# Detect environment: Colab or local
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # Install navis on Colab
    import subprocess
    subprocess.run(["pip", "install", "navis", "-q"], check=True)
    # Clone the repo so all demo data is available
    if not Path("mixed-polarity-paper").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/deutschlab/mixed-polarity-paper.git"], check=True)
    DEMO_DIR = Path("mixed-polarity-paper/demo")
else:
    # Running locally: notebook lives inside demo/
    DEMO_DIR = Path(".").resolve()

TABLES_DIR = DEMO_DIR
SWC_DIR    = DEMO_DIR / "swc"
OUTPUT_DIR = DEMO_DIR / "output"


print("DEMO_DIR:", DEMO_DIR)
print("Tables:", [p.name for p in TABLES_DIR.iterdir()])

In [ ]:
NodesG = pd.read_csv(TABLES_DIR / "Supplemental_file1_neuron_annotations_demo.csv")
allsynapses = pd.read_csv(TABLES_DIR / "fafb_v783_princeton_synapse_table_demo.csv")

In [ ]:
allsynapses = allsynapses.reset_index(drop=False)
allsynapses = allsynapses.drop(columns=['ctr_x', 'ctr_y', 'ctr_z'])

allsynapses["pre_root_id_720575940"] = (
    720575940 * 10**allsynapses["pre_root_id_720575940"].astype(str).str.len()
    + allsynapses["pre_root_id_720575940"].astype(int)
).astype(np.int64)

allsynapses["post_root_id_720575940"] = (
    720575940 * 10**allsynapses["post_root_id_720575940"].astype(str).str.len()
    + allsynapses["post_root_id_720575940"].astype(int)
).astype(np.int64)

In [ ]:
allsynapses

In [ ]:
allsynapses.columns = ['synapse_id', 'pre_x', 'pre_y', 'pre_z', 'post_x',
                        'post_y', 'post_z', 'size', 'pre',
                        'post', 'neuropil']

In [ ]:
import navis

def get_split(item, flow_thresh=1):
    try:
        split = navis.split_axon_dendrite(item, metric='synapse_flow_centrality',
                                           reroot_soma=True, flow_thresh=flow_thresh)
        return split
    except:
        return 'split issue'

def parents_check_on_comp(compartment):
    parents = compartment.nodes[compartment.nodes['parent_id'] == -1]
    return len(parents) == 1

def SI_calc(neuron_id_data):
    ax_pre    = neuron_id_data[1][0]
    ax_post   = neuron_id_data[1][1]
    dend_pre  = neuron_id_data[2][0]
    dend_post = neuron_id_data[2][1]
    ent_ax    = calc_s(ax_pre, ax_post)
    ent_dend  = calc_s(dend_pre, dend_post)
    snorm     = calc_s(ax_pre + dend_pre, ax_post + dend_post)
    S  = ((ax_pre + ax_post) * ent_ax + (dend_pre + dend_post) * ent_dend) / \
         (ax_pre + ax_post + dend_pre + dend_post)
    SI = 1 - (S / snorm)
    IG = snorm - S
    return SI, IG

def attach_synapses_princeton(x, syn, neuropils=False, batch_size=100, progress=True):
    if isinstance(x, navis.core.BaseNeuron):
        x = navis.NeuronList([x])
    if isinstance(x, navis.NeuronList):
        for n in x:
            add_cols = ['neuropil'] if neuropils else []
            cols = ['pre_x', 'pre_y', 'pre_z', 'post', 'synapse_id', 'size'] + add_cols
            presyn = syn.loc[syn.pre == np.int64(n.id), cols].rename(
                {'pre_x': 'x', 'pre_y': 'y', 'pre_z': 'z', 'post': 'partner_id'}, axis=1)
            presyn['type'] = 'pre'
            cols = ['post_x', 'post_y', 'post_z', 'pre', 'synapse_id', 'size'] + add_cols
            postsyn = syn.loc[syn.post == np.int64(n.id), cols].rename(
                {'post_x': 'x', 'post_y': 'y', 'post_z': 'z', 'pre': 'partner_id'}, axis=1)
            postsyn['type'] = 'post'
            connectors = pd.concat((presyn, postsyn), axis=0, ignore_index=True)
            connectors['type'] = connectors['type'].astype('category')
            if isinstance(n, navis.TreeNeuron):
                tree = navis.neuron2KDTree(n, data='nodes')
                dist, ix = tree.query(connectors[['x', 'y', 'z']].values)
                too_far = dist > 10_000
                if any(too_far):
                    connectors = connectors[~too_far].copy()
                    ix = ix[~too_far]
                connectors['node_id'] = n.nodes.node_id.values[ix]
                connectors.insert(0, 'connector_id', np.arange(connectors.shape[0]))
            n.connectors = connectors
    return x

In [ ]:
allsynapses = allsynapses[allsynapses['pre'] != allsynapses['post']]

In [ ]:
swc_list = navis.read_swc(str(SWC_DIR))

In [ ]:
issues_in_comp = []
linker_neurons = []
all_SI = []
all_non_succsessfull_connectors = pd.DataFrame(columns=[
    'connector_id', 'x', 'y', 'z', 'partner_id', 'type',
    'node_id', 'compartment', 'neuron', 'synapse_id', 'size'])
all_connectors = pd.DataFrame(columns=[
    'connector_id', 'x', 'y', 'z', 'partner_id', 'type',
    'node_id', 'compartment', 'neuron', 'synapse_id', 'size'])

In [ ]:
swc_list_id = [np.int64(i.id) for i in swc_list]
allsynapses2 = allsynapses[
    (allsynapses['pre'].isin(swc_list_id)) | (allsynapses['post'].isin(swc_list_id))]

swc_list_to_analyse = navis.NeuronList(swc_list[0:])
healed_attached_neurons_list = attach_synapses_princeton(swc_list_to_analyse, allsynapses2)
num_items = len(healed_attached_neurons_list)

In [ ]:
from math import log

def calc_s(pre, post):
    total = pre + post
    if total > 0:
        p_pre  = pre  / total
        p_post = post / total
        entropy_pre  = p_pre  * log(p_pre,  2) if p_pre  > 0 else 0
        entropy_post = p_post * log(p_post, 2) if p_post > 0 else 0
        entropy = -(entropy_pre + entropy_post)
    else:
        entropy = 0
    return entropy

In [ ]:
import time
start_time = time.time()

for i in range(0, num_items):
    print(f'--- {i} / {num_items} ---')
    n = healed_attached_neurons_list[i]

    if len(n.nodes) > 80000:
        issues_in_comp.append([n.id, len(n.nodes)])
        continue

    axon_swc = dend_swc = linker_swc = None
    linker = False
    split = get_split(n)

    if len(split) == 3:
        if split[0].compartment == 'dendrite' and split[2].compartment == 'axon':
            axon_swc, linker_swc, dend_swc = split[2], split[1], split[0]
            linker = True
            linker_neurons.append(n.id)
        elif split[2].compartment == 'dendrite' and split[0].compartment == 'axon':
            axon_swc, linker_swc, dend_swc = split[0], split[1], split[2]
            linker = True
            linker_neurons.append(n.id)
        else:
            issues_in_comp.append([n.id, 'issue1'])
            all_non_succsessfull_connectors = pd.concat(
                [all_non_succsessfull_connectors, split.connectors])
            continue
    elif len(split) == 2:
        if split[0].compartment == 'dendrite' and split[1].compartment == 'axon':
            axon_swc, dend_swc = split[1], split[0]
        elif split[1].compartment == 'dendrite' and split[0].compartment == 'axon':
            axon_swc, dend_swc = split[0], split[1]
        else:
            issues_in_comp.append([n.id, 'issue2'])
            all_non_succsessfull_connectors = pd.concat(
                [all_non_succsessfull_connectors, split.connectors])
            continue
    else:
        issues_in_comp.append([n.id, 'issue3'])
        continue

    SI, IG = SI_calc(['',
                      (len(axon_swc.presynapses), len(axon_swc.postsynapses)),
                      (len(dend_swc.presynapses), len(dend_swc.postsynapses))])
    all_SI.append([n.id, SI])

    axon_connectors = axon_swc.connectors.copy()
    axon_connectors['compartment'] = 'A'
    dend_connectors = dend_swc.connectors.copy()
    dend_connectors['compartment'] = 'D'

    if linker:
        linker_connectors = linker_swc.connectors.copy()
        linker_connectors['compartment'] = 'L'
        neuron_connectors = pd.concat([axon_connectors, dend_connectors, linker_connectors])
    else:
        neuron_connectors = pd.concat([axon_connectors, dend_connectors])

    neuron_connectors['neuron'] = n.id
    neuron_connectors = neuron_connectors.reset_index(drop=True)
    all_connectors = pd.concat([all_connectors, neuron_connectors])

elapsed = time.time() - start_time
print(f'Done in {elapsed:.1f}s  |  Successful: {len(all_SI)}  |  Issues: {len(issues_in_comp)}')

In [ ]:
SI_df = pd.DataFrame(all_SI, columns=['root_id', 'SI'])
print(SI_df.describe())
SI_df.head(10)

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SI_df.to_csv(OUTPUT_DIR / 'SI_results_demo.csv', index=False)
all_connectors.to_csv(OUTPUT_DIR / 'connectors_demo.csv', index=False)
print('Saved to', OUTPUT_DIR)